# Forecast Accuracy Metrics

How do we know if a forecast is "good"?  We need **numerical measures of
accuracy** that quantify the gap between forecasts and actuals.  This
notebook covers the most important metrics, their strengths and weaknesses,
and the critical rule for time series evaluation: **never use random splits**.

## Three families of error metrics

| Family | Metrics | Pros | Cons |
|---|---|---|---|
| Scale-dependent | MAE, MSE, RMSE | Easy to interpret in original units | Cannot compare across series with different scales |
| Percentage | MAPE | Scale-free | Undefined when $y_t = 0$; asymmetric |
| Scaled | MASE | Scale-free; well-defined for all data | Requires a naive benchmark |

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# Load airline passengers
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

# Train/test split: first 120 months train, last 24 test
TRAIN_SIZE = 120
train = airline.iloc[:TRAIN_SIZE]
test = airline.iloc[TRAIN_SIZE:]

print(f"Train: {len(train)} obs, Test: {len(test)} obs")
print(f"Train period: {train.index[0].date()} to {train.index[-1].date()}")
print(f"Test period:  {test.index[0].date()} to {test.index[-1].date()}")

---
## 1. Train/Test Split for Time Series

**Critical rule: NEVER use random train/test splits for time series data.**

Why not?  Because time series observations are dependent — if you put
March 2020 in the test set but keep February and April 2020 in training,
you are "peeking" at the future.  The model can exploit temporal proximity
to score well on the test set without actually being able to forecast.

**Correct approach**: split by time.  Training set = earlier observations,
test set = later observations.  Rule of thumb: 80/20 split, or reserve the
last year/quarter as the test set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# WRONG: random split
np.random.seed(42)
random_idx = np.random.permutation(len(airline))
random_train_idx = sorted(random_idx[:TRAIN_SIZE])
random_test_idx = sorted(random_idx[TRAIN_SIZE:])

axes[0].plot(airline.index[random_train_idx],
             airline["Passengers"].values[random_train_idx],
             "o", markersize=3, label="Train", alpha=0.7)
axes[0].plot(airline.index[random_test_idx],
             airline["Passengers"].values[random_test_idx],
             "o", markersize=3, label="Test", color="tab:orange", alpha=0.7)
axes[0].set_title("WRONG: Random Split (interleaved)")
axes[0].legend()

# RIGHT: temporal split
axes[1].plot(train["Passengers"], label="Train")
axes[1].plot(test["Passengers"], label="Test", color="tab:orange")
axes[1].axvline(test.index[0], color="grey", linestyle="--", alpha=0.5)
axes[1].set_title("CORRECT: Temporal Split")
axes[1].legend()

for ax in axes:
    ax.set_ylabel("Passengers")
    ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

print("Random splits allow the model to 'see' future data during training.")
print("Temporal splits properly simulate real forecasting conditions.")

---
## 2. Generate Forecasts from All Four Benchmark Methods

In [ ]:
HORIZON = len(test)
actual = test["Passengers"].values


def forecast_mean(train_s, h):
    return np.full(h, train_s.mean())


def forecast_naive(train_s, h):
    return np.full(h, train_s.iloc[-1])


def forecast_seasonal_naive(train_s, h, m=12):
    last_season = train_s.values[-m:]
    return np.tile(last_season, int(np.ceil(h / m)))[:h]


def forecast_drift(train_s, h):
    slope = (train_s.iloc[-1] - train_s.iloc[0]) / (len(train_s) - 1)
    return np.array([train_s.iloc[-1] + i * slope for i in range(1, h + 1)])


forecasts = {
    "Mean": forecast_mean(train["Passengers"], HORIZON),
    "Naive": forecast_naive(train["Passengers"], HORIZON),
    "Seasonal Naive": forecast_seasonal_naive(train["Passengers"], HORIZON),
    "Drift": forecast_drift(train["Passengers"], HORIZON),
}

for name, fc in forecasts.items():
    print(f"{name:15s}: first 3 values = {fc[:3].round(1)}")

---
## 3. Scale-Dependent Metrics

### Mean Absolute Error (MAE)

$$
\text{MAE} = \frac{1}{h}\sum_{t=1}^{h} |y_t - \hat{y}_t|
$$

The average magnitude of the forecast errors.  Easy to understand:
"on average, our forecast is off by X units."

In [ ]:
def mae(actual, predicted):
    """Mean Absolute Error — from scratch."""
    return np.mean(np.abs(actual - predicted))


# Verify against sklearn
our_mae = mae(actual, forecasts["Naive"])
sk_mae = mean_absolute_error(actual, forecasts["Naive"])

print(f"Our MAE:     {our_mae:.4f}")
print(f"sklearn MAE: {sk_mae:.4f}")
print(f"Match: {np.isclose(our_mae, sk_mae)}")

### Mean Squared Error (MSE) and Root Mean Squared Error (RMSE)

$$
\text{MSE} = \frac{1}{h}\sum_{t=1}^{h} (y_t - \hat{y}_t)^2
$$

$$
\text{RMSE} = \sqrt{\text{MSE}}
$$

MSE penalises large errors more heavily than MAE (due to squaring).
RMSE brings the units back to the original scale.

In [ ]:
def mse(actual, predicted):
    """Mean Squared Error — from scratch."""
    return np.mean((actual - predicted) ** 2)


def rmse(actual, predicted):
    """Root Mean Squared Error — from scratch."""
    return np.sqrt(mse(actual, predicted))


# Verify against sklearn
our_mse = mse(actual, forecasts["Naive"])
sk_mse = mean_squared_error(actual, forecasts["Naive"])

print(f"Our MSE:     {our_mse:.4f}")
print(f"sklearn MSE: {sk_mse:.4f}")
print(f"Our RMSE:    {rmse(actual, forecasts['Naive']):.4f}")
print(f"sklearn RMSE: {np.sqrt(sk_mse):.4f}")

In [ ]:
# Why RMSE > MAE: RMSE penalises large errors more
print("MAE vs RMSE for each method:")
print(f"{'Method':20s} {'MAE':>8s} {'RMSE':>8s} {'RMSE/MAE':>10s}")
print("-" * 48)
for name, fc in forecasts.items():
    m = mae(actual, fc)
    r = rmse(actual, fc)
    print(f"{name:20s} {m:8.2f} {r:8.2f} {r/m:10.2f}")
print()
print("RMSE/MAE ratio > 1 always. Larger ratio = more large errors.")

---
## 4. Percentage Errors

### Mean Absolute Percentage Error (MAPE)

$$
\text{MAPE} = \frac{100}{h}\sum_{t=1}^{h} \left|\frac{y_t - \hat{y}_t}{y_t}\right|
$$

MAPE is scale-free, so you can use it to compare forecasts across different
series.  However, it has two serious problems:

1. **Undefined when $y_t = 0$** — dividing by zero.
2. **Asymmetric** — it penalises positive errors (under-forecasting) more
   than negative errors (over-forecasting) of the same magnitude.

In [ ]:
def mape(actual, predicted):
    """Mean Absolute Percentage Error — from scratch."""
    return 100 * np.mean(np.abs((actual - predicted) / actual))


# Verify against sklearn (sklearn returns proportion, not percentage)
our_mape = mape(actual, forecasts["Naive"])
sk_mape = mean_absolute_percentage_error(actual, forecasts["Naive"]) * 100

print(f"Our MAPE:     {our_mape:.4f}%")
print(f"sklearn MAPE: {sk_mape:.4f}%")
print(f"Match: {np.isclose(our_mape, sk_mape)}")

In [ ]:
# Demonstrate MAPE's asymmetry problem
y_actual = np.array([100.0])

# Over-predict by 50
over = np.array([150.0])
# Under-predict by 50
under = np.array([50.0])

print("Same absolute error (50 units), different MAPE:")
print(f"  Actual=100, Forecast=150 (over):  MAPE = {mape(y_actual, over):.1f}%")
print(f"  Actual=100, Forecast=50 (under):  MAPE = {mape(y_actual, under):.1f}%")
print()
print("Under-forecasting gets a MAPE of 50%, over-forecasting only 50%.")
print("But try actual=50:")
y2 = np.array([50.0])
print(f"  Actual=50, Forecast=100 (over by 50): MAPE = {mape(y2, np.array([100.0])):.1f}%")
print(f"  Actual=50, Forecast=0 (under by 50):  MAPE = {mape(y2, np.array([0.001])):.1f}%")
print()
print("MAPE heavily penalises errors on small actuals — it can be very misleading.")

---
## 5. Scaled Errors: MASE

### Mean Absolute Scaled Error (MASE)

The MASE normalises errors by the MAE of the **naive method** (for
non-seasonal data) or **seasonal naive** (for seasonal data) applied
in-sample:

$$
\text{MASE} = \frac{\text{MAE of model}}{\text{MAE of in-sample naive forecast}}
= \frac{\frac{1}{h}\sum_{t=1}^{h}|y_t - \hat{y}_t|}{\frac{1}{T-1}\sum_{t=2}^{T}|y_t - y_{t-1}|}
$$

**Interpretation:**
- MASE < 1: the model beats the naive benchmark
- MASE = 1: the model performs exactly like naive
- MASE > 1: the model is *worse* than naive

In [ ]:
def mase(actual, predicted, train_series, seasonal_period=1):
    """
    Mean Absolute Scaled Error.

    Parameters
    ----------
    actual : array-like
        Test set actual values.
    predicted : array-like
        Forecast values.
    train_series : array-like
        Training set values (used to compute naive MAE).
    seasonal_period : int
        1 for non-seasonal naive, m for seasonal naive.
    """
    train_vals = np.array(train_series)
    # In-sample naive errors
    naive_errors = np.abs(train_vals[seasonal_period:] - train_vals[:-seasonal_period])
    naive_mae = np.mean(naive_errors)

    # Forecast errors
    forecast_mae = np.mean(np.abs(actual - predicted))

    return forecast_mae / naive_mae


# Compute MASE for each method
train_vals = train["Passengers"].values

print("MASE (scaled by in-sample naive MAE):")
for name, fc in forecasts.items():
    m = mase(actual, fc, train_vals, seasonal_period=1)
    print(f"  {name:20s}: MASE = {m:.4f}  {'(beats naive)' if m < 1 else '(worse than naive)'}")

In [ ]:
# MASE scaled by seasonal naive (more appropriate for seasonal data)
print("MASE (scaled by in-sample SEASONAL naive MAE, m=12):")
for name, fc in forecasts.items():
    m = mase(actual, fc, train_vals, seasonal_period=12)
    print(f"  {name:20s}: MASE = {m:.4f}  {'(beats snaive)' if m < 1 else '(worse than snaive)'}")

---
## 6. All Metrics Summary Table

Let us compute all metrics for all four methods and display a clean
comparison table.

In [ ]:
def compute_all_metrics(actual, predicted, train_series, seasonal_period=12):
    """Compute all standard forecast accuracy metrics."""
    return {
        "MAE": mae(actual, predicted),
        "MSE": mse(actual, predicted),
        "RMSE": rmse(actual, predicted),
        "MAPE (%)": mape(actual, predicted),
        "MASE": mase(actual, predicted, train_series, seasonal_period),
    }


rows = []
for name, fc in forecasts.items():
    metrics = compute_all_metrics(actual, fc, train_vals)
    metrics["Method"] = name
    rows.append(metrics)

results = pd.DataFrame(rows)[["Method", "MAE", "RMSE", "MAPE (%)", "MASE"]]
results = results.round(2)
print(results.to_string(index=False))

In [ ]:
# Visual comparison of metrics
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

metrics_to_plot = ["MAE", "RMSE", "MAPE (%)", "MASE"]
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

for ax, metric in zip(axes, metrics_to_plot):
    vals = results.set_index("Method")[metric]
    bars = ax.bar(vals.index, vals, color=colors, alpha=0.7, edgecolor="black")
    ax.set_title(metric, fontsize=13)
    ax.tick_params(axis="x", rotation=45)

    # Highlight best (lowest)
    best_idx = vals.idxmin()
    for bar, name in zip(bars, vals.index):
        if name == best_idx:
            bar.set_edgecolor("black")
            bar.set_linewidth(2)

plt.suptitle("Metric Comparison Across Benchmark Methods", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Visualising Forecast Errors

Looking at the raw errors (not just summary metrics) reveals *when* each
method does well or poorly.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, fc), color in zip(
    axes.flat,
    forecasts.items(),
    ["tab:red", "tab:green", "tab:purple", "tab:brown"],
):
    errors = actual - fc
    ax.bar(test.index, errors, color=color, alpha=0.7, width=20)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"{name} — Forecast Errors")
    ax.set_ylabel("Error (actual - forecast)")

plt.tight_layout()
plt.show()

print("Positive errors = model under-predicted; Negative = over-predicted.")
print("Seasonal naive has the smallest and most balanced errors.")

---
## 8. Implementing with sklearn

`sklearn.metrics` provides several of these metrics out of the box.

In [ ]:
# sklearn implementations
fc = forecasts["Seasonal Naive"]

print("sklearn metrics for Seasonal Naive:")
print(f"  MAE:  {mean_absolute_error(actual, fc):.2f}")
print(f"  MSE:  {mean_squared_error(actual, fc):.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(actual, fc)):.2f}")
print(f"  MAPE: {mean_absolute_percentage_error(actual, fc) * 100:.2f}%")
print()
print("Note: sklearn does not have MASE built-in — use our custom function.")

---
## 9. When to Use Which Metric

| Metric | Use when... | Avoid when... |
|---|---|---|
| **MAE** | Comparing methods on the same series; all errors matter equally | Comparing across series of different scales |
| **RMSE** | Large errors are particularly costly | You want robustness to outliers |
| **MAPE** | You want a percentage interpretation; comparing across series | Series has zeros or near-zero values |
| **MASE** | Comparing across series; benchmarking against naive | (Always appropriate) |

**Recommendation**: Always report MASE alongside one scale-dependent metric
(MAE or RMSE).  Avoid relying on MAPE alone.

In [ ]:
# Demonstrate MAPE failure with near-zero values
y_with_zeros = np.array([100, 200, 0.1, 150, 300])
fc_example = np.array([110, 190, 5, 140, 310])

print("Example with near-zero actual value:")
print(f"  Actuals:   {y_with_zeros}")
print(f"  Forecasts: {fc_example}")
print(f"  MAE:  {mae(y_with_zeros, fc_example):.2f}")
print(f"  MAPE: {mape(y_with_zeros, fc_example):.1f}%")
print()
print("The MAPE is dominated by the near-zero observation (0.1 vs 5).")
print("An error of 4.9 units produces a 4900% percentage error!")
print("This single observation inflates the average MAPE enormously.")

---
## 10. Multi-Step vs Single-Step Accuracy

Forecast accuracy typically **degrades with the forecast horizon**.  It is
useful to plot the error as a function of horizon $h$.

In [ ]:
# Error by horizon for each method
fig, ax = plt.subplots(figsize=(12, 5))

for name, fc, color in zip(
    forecasts.keys(),
    forecasts.values(),
    ["tab:red", "tab:green", "tab:purple", "tab:brown"],
):
    abs_errors = np.abs(actual - fc)
    ax.plot(range(1, HORIZON + 1), abs_errors, label=name, color=color, marker="o", markersize=4)

ax.set_xlabel("Forecast Horizon (months ahead)")
ax.set_ylabel("Absolute Error")
ax.set_title("Absolute Error by Forecast Horizon")
ax.legend()
plt.tight_layout()
plt.show()

print("Errors generally grow with horizon for mean, naive, and drift.")
print("Seasonal naive errors oscillate with the seasonal pattern.")

In [ ]:
# Cumulative MAE by horizon
fig, ax = plt.subplots(figsize=(12, 5))

for name, fc, color in zip(
    forecasts.keys(),
    forecasts.values(),
    ["tab:red", "tab:green", "tab:purple", "tab:brown"],
):
    abs_errors = np.abs(actual - fc)
    cumulative_mae = np.cumsum(abs_errors) / np.arange(1, HORIZON + 1)
    ax.plot(range(1, HORIZON + 1), cumulative_mae, label=name, color=color, linewidth=2)

ax.set_xlabel("Forecast Horizon (months ahead)")
ax.set_ylabel("Cumulative MAE")
ax.set_title("Cumulative MAE by Forecast Horizon")
ax.legend()
plt.tight_layout()
plt.show()

---
## 11. Reusable Accuracy Comparison Function

In [ ]:
def accuracy_table(actual, forecasts_dict, train_series, seasonal_period=12):
    """
    Compute MAE, RMSE, MAPE, and MASE for a dictionary of forecasts.

    Parameters
    ----------
    actual : array-like
        Test set actual values.
    forecasts_dict : dict
        {method_name: forecast_array}.
    train_series : array-like
        Training data (for MASE denominator).
    seasonal_period : int
        Seasonal period for MASE.

    Returns
    -------
    pd.DataFrame
    """
    rows = []
    for name, fc in forecasts_dict.items():
        rows.append({
            "Method": name,
            "MAE": mae(actual, fc),
            "RMSE": rmse(actual, fc),
            "MAPE (%)": mape(actual, fc),
            "MASE": mase(actual, fc, train_series, seasonal_period),
        })
    df = pd.DataFrame(rows).set_index("Method").round(2)
    return df.sort_values("MAE")


table = accuracy_table(actual, forecasts, train_vals)
print(table)

In [ ]:
# Apply to a second dataset for comparison
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.index.freq = "MS"
alcohol.columns = ["Sales"]
alcohol = alcohol.dropna()

alc_train = alcohol.iloc[:-24]
alc_test = alcohol.iloc[-24:]
alc_h = len(alc_test)
alc_actual = alc_test["Sales"].values

alc_forecasts = {
    "Mean": forecast_mean(alc_train["Sales"], alc_h),
    "Naive": forecast_naive(alc_train["Sales"], alc_h),
    "Seasonal Naive": forecast_seasonal_naive(alc_train["Sales"], alc_h),
    "Drift": forecast_drift(alc_train["Sales"], alc_h),
}

print("Alcohol Sales — Forecast Accuracy")
print(accuracy_table(alc_actual, alc_forecasts, alc_train["Sales"].values))

---
## Summary

- **MAE, MSE, RMSE** measure accuracy in original units — useful for
  comparing methods on the *same* series.
- **MAPE** gives a percentage but is **undefined at zero** and **asymmetric**
  — use with caution.
- **MASE** normalises by the naive benchmark — values below 1.0 mean the
  model beats naive.  It is the most robust metric for general use.
- **Time series train/test splits must respect temporal order** — never
  use random splits.
- Always compute **multiple metrics** and examine errors by horizon to get
  a complete picture of forecast quality.
- The `accuracy_table()` function defined here can be reused throughout
  subsequent chapters.

In [ ]:
print("Next notebook: Time Series Cross-Validation — a more robust way")
print("to estimate out-of-sample accuracy than a single train/test split.")